In [3]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# --------------------------
# 1. Load Dataset
# --------------------------
df = pd.read_csv("hf://datasets/KathiS/new_preprocessed_dataset/new_cleaned.csv")
print("Loaded dataset:", df.shape)

# --------------------------
# 2. Select Important Features
# --------------------------
important_features = [
    "Flow_Duration",
    "Init_Bwd_Win_Byts",
    "Dst_Port",
    "Idle_Mean",
    "Flow_IAT_Min",
    "Flow_Pkts/s",
    "Pkt_Len_Max",
    "ACK_Flag_Cnt",
    "Fwd_Pkt_Len_Max",
    "Bwd_Header_Len",
    "TotLen_Bwd_Pkts",
    "Fwd_Pkts/s",
    "Flow_Byts/s",
    "Bwd_Pkt_Len_Max",
    "Bwd_Pkts/s",
    "TotLen_Fwd_Pkts",
    "Flow_IAT_Max",
    "Fwd_Pkt_Len_Min",
    "Bwd_Pkt_Len_Mean",
    "Pkt_Len_Std",
    "SYN_Flag_Cnt",
    "Pkt_Len_Mean"
]

label_column = "Label"
final_columns = important_features + [label_column]

df_selected = df[final_columns]
print("Shape after feature selection:", df_selected.shape)

# --------------------------
# 3. Separate Features and Label
# --------------------------
X = df_selected[important_features]
y = df_selected[label_column]

# --------------------------
# 4. Balanced Sampling (Under + SMOTE)
# --------------------------
print("Applying balancing...")

pipeline = Pipeline([
    ("under", RandomUnderSampler(sampling_strategy='auto')),
    ("smote", SMOTE(sampling_strategy='auto'))
])

X_bal, y_bal = pipeline.fit_resample(X, y)

print("Balanced shape:", X_bal.shape)
print("Class distribution after balancing:")
print(y_bal.value_counts())

# --------------------------
# 5. Scaling
# --------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_bal)

# Create final sampled dataframe
final_df = pd.DataFrame(X_scaled, columns=important_features)
final_df["Label"] = y_bal.values

print("\nFinal dataset (after sampling & scaling):", final_df.shape)

# --------------------------
# 6. Decision Tree Plots Using Sampled Data
# --------------------------
print("\nGenerating Decision Tree plots using sampled dataset...")

# Identify top 3 attack categories AFTER SAMPLING
top3_attacks = final_df["Label"].value_counts().head(3).index.tolist()
print("Top 3 attacks:", top3_attacks)

# Save to Downloads
downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")

for attack in top3_attacks:
    print(f"\nTraining Decision Tree for major attack: {attack}")

    # Binary dataset: attack vs all
    df_binary = final_df.copy()
    df_binary["BinaryLabel"] = (df_binary["Label"] == attack).astype(int)

    X_bin = df_binary[important_features]
    y_bin = df_binary["BinaryLabel"]

    # Train Decision Tree
    dt = DecisionTreeClassifier(max_depth=4)
    dt.fit(X_bin, y_bin)

    # Plot decision tree
    plt.figure(figsize=(22, 14))
    plot_tree(
        dt,
        feature_names=important_features,
        class_names=["Other", attack],
        filled=True,
        fontsize=8
    )

    # Save to Downloads
    file_path = os.path.join(downloads_path, f"{attack}_Decision_Tree.png")
    plt.savefig(file_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Saved → {file_path}")

# --------------------------
# 7. Save Final Processed Dataset
# --------------------------
csv_path = os.path.join(downloads_path, "Final_Preprocessed.csv")
final_df.to_csv(csv_path, index=False)

print(f"\nFinal processed dataset saved to: {csv_path}")


Loaded dataset: (625783, 70)
Shape after feature selection: (625783, 23)
Applying balancing...
Balanced shape: (199728, 22)
Class distribution after balancing:
Label
DoS-Synflooding          22192
MITM ARP Spoofing        22192
Mirai-Ackflooding        22192
Mirai-HTTP Flooding      22192
Mirai-Hostbruteforceg    22192
Mirai-UDP Flooding       22192
Normal                   22192
Scan Hostport            22192
Scan Port OS             22192
Name: count, dtype: int64

Final dataset (after sampling & scaling): (199728, 23)

Generating Decision Tree plots using sampled dataset...
Top 3 attacks: ['DoS-Synflooding', 'MITM ARP Spoofing', 'Mirai-Ackflooding']

Training Decision Tree for major attack: DoS-Synflooding
Saved → C:\Users\Hp\Downloads\DoS-Synflooding_Decision_Tree.png

Training Decision Tree for major attack: MITM ARP Spoofing
Saved → C:\Users\Hp\Downloads\MITM ARP Spoofing_Decision_Tree.png

Training Decision Tree for major attack: Mirai-Ackflooding
Saved → C:\Users\Hp\Downloads\M